# Speech recognition: a small demo

I didn't have time to set up a microphone pipeline, so this notebook runs on recorded speech with reference transcripts: the first 20 clips of the LibriSpeech "clean" sample that Hugging Face keeps for testing (`hf-internal-testing/librispeech_asr_dummy`, 73 clips of real read audiobook speech; LibriSpeech is CC BY 4.0). Two paths:

1. The `speech_recognition` package as a thin wrapper around Google's Web Speech API (needs internet).
2. OpenAI's `whisper` package with the `tiny` model for an offline option (downloads about 75MB the first time).

Because the clips come with transcripts, I can report word error rate (WER) instead of just printing strings.

In [1]:
# !pip install SpeechRecognition openai-whisper -q  # OR transformers+torch for whisper

In [2]:
from pathlib import Path
import io, re
import numpy as np
import pandas as pd
import soundfile as sf

DATA = Path("../../data/librispeech_sample")
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

PQ = DATA / "librispeech_dummy_clean.parquet"
if not PQ.exists():
    pd.read_parquet("https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy/"
                    "resolve/main/clean/validation-00000-of-00001.parquet").to_parquet(PQ)
clips = pd.read_parquet(PQ).head(20)

# write each clip as 16 kHz 16-bit PCM WAV, the format speech_recognition reads reliably
have_audio, refs = [], {}
for _, r in clips.iterrows():
    audio, sr_ = sf.read(io.BytesIO(r["audio"]["bytes"]))
    out = DATA / f"{r['id']}.wav"
    sf.write(out, audio, sr_, subtype="PCM_16")
    have_audio.append(out); refs[out.name] = r["text"]
print(len(have_audio), "clips, total", round(sum(sf.info(p).duration for p in have_audio), 1), "seconds")
print("example reference:", refs[have_audio[0].name])

def words(s):
    return re.sub(r"[^a-z' ]", " ", s.lower()).split()

def wer(ref, hyp):
    r, h = words(ref), words(hyp)
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = np.arange(len(r) + 1); d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    return d[len(r), len(h)], len(r)    # (word edits, reference words)

20 clips, total 187.7 seconds
example reference: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL


## Path A: SpeechRecognition + Google

Simple, but requires internet and is rate-limited.

In [3]:
try:
    import speech_recognition as sr
    HAVE_SR = True
except Exception:
    HAVE_SR = False
    print("speech_recognition package not installed, skipping path A")

if HAVE_SR:
    r = sr.Recognizer()
    edits = total = 0
    for p in have_audio[:5]:          # the free endpoint is rate-limited, so only 5 clips
        with sr.AudioFile(str(p)) as src:
            audio = r.record(src)
        try:
            text = r.recognize_google(audio)
        except sr.UnknownValueError:
            text = ""
        except Exception as e:
            print(f"[google] {p.name} -> error: {repr(e)[:100]}"); continue
        e, n = wer(refs[p.name], text); edits += e; total += n
        print(f"[google] {p.name:>22} -> {text!r}")
    if total:
        print(f"\nGoogle Web Speech WER on {min(5, len(have_audio))} clips: {edits / total:.3f}")

[google]   1272-128104-0000.wav -> 'Mr quilter is the Apostle of the middle classes and we are glad to welcome his gospel'


[google]   1272-128104-0001.wav -> "nor is Mr Quilter's mannerless interesting than his matter"


[google]   1272-128104-0002.wav -> 'can you tell us that at this festive season of the year with Christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind'


[google]   1272-128104-0003.wav -> "he has Grave doubts whether Sir Frederic Layton's work is really Greek after all and can discover in it but little of Rocky Ithaca"


[google]   1272-128104-0004.wav -> "lynels pictures are a sort of up guards and atom paintings and Mason's Exquisite Idols are as National as a Jingo poem Mr birkett Foster's Landscapes smile at one much in the same way that Mr Carter used to flash his teeth and Mr John Collier gives his sitter a cheerful slap in the back before he says like a shampooer and a Turkish bath next man"

Google Web Speech WER on 5 clips: 0.132


## Path B: whisper-tiny (offline after first download)

Multilingual, robust, and the smallest Whisper variant is small enough to run on a laptop CPU. The transcription latency for a 3-second clip is maybe a second or two.

In [4]:
HAVE_WHISPER = False
try:
    import whisper
    wmodel = whisper.load_model("tiny")
    HAVE_WHISPER = True
except Exception as e:
    print("whisper not available:", repr(e)[:120])

if HAVE_WHISPER:
    edits = total = 0
    for i, p in enumerate(have_audio):
        # fp16=False because we're on CPU and the default complains.
        out = wmodel.transcribe(str(p), fp16=False, language="en")
        e, n = wer(refs[p.name], out["text"]); edits += e; total += n
        if i < 3:
            print(f"[whisper-tiny] ref: {refs[p.name]}\n               hyp: {out['text'].strip()}\n")
    print(f"whisper-tiny WER on {len(have_audio)} clips ({total} reference words): {edits / total:.3f}")

[whisper-tiny] ref: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
               hyp: Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel.



[whisper-tiny] ref: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
               hyp: Nor is Mr. Quilters' manner less interesting than his matter.



[whisper-tiny] ref: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
               hyp: He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similarly drawn from eating and its results occur most readily to the mind.



whisper-tiny WER on 20 clips (448 reference words): 0.112


## Notes

A few honest things:

- On 20 real LibriSpeech clips (188 seconds, 448 reference words), whisper-tiny gets a word error rate of 0.112. Google's free Web Speech endpoint gets 0.132 on the first 5 clips (only 5 because it's rate-limited, so that number is shakier).
- Part of both error rates is just text normalization. LibriSpeech references spell out "MISTER" while both systems write "Mr", and my simple WER counts that as an error. Proper evaluation would run both sides through the same normalizer (Whisper ships one) before scoring.
- The real misses are easy to spot in the outputs: Google wrote "can you tell us" for "he tells us", Whisper wrote "similarly drawn" for "similes drawn". Read audiobook speech is also the easy case. Phone audio, accents and background noise would push these numbers up a lot.
- Real deployment would need a streaming mic loop (PyAudio / sounddevice) and probably a voice activity detector to chunk speech. Not in scope for a weekend notebook.